In [3]:
# %pip install -q --upgrade pip
# %pip install -q --upgrade tensorflow tensorflow-datasets transformers accelerate evaluate
# Il est recommandé de redémarrer l'environnement d'exécution (Runtime -> Restart runtime) après cette installation.

In [4]:
# import transformers

# print(transformers.__version__)
# print(hasattr(transformers, "TFBertForSequenceClassification"))
# print(hasattr(transformers, "BertTokenizer"))

# %pip install -U transformers

import platform
import transformers
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

# Make the dataset mapping run eagerly so the tokenizer can execute reliably in this environment.
tf.config.run_functions_eagerly(True)


print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

c:\Users\meles\Documents\TTA_DI_BootCamp_Gilles-Chris_MAKE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Python version      : 3.12.10
TensorFlow version  : 2.21.0
GPU devices detected: []


In [5]:
import tensorflow_datasets as tfds

print(tfds)
print("Version :", tfds.__version__)
print("load existe :", hasattr(tfds, "load"))
print(dir(tfds)[:20])

<module 'tensorflow_datasets' from 'c:\\Users\\meles\\Documents\\TTA_DI_BootCamp_Gilles-Chris_MAKE\\.venv\\Lib\\site-packages\\tensorflow_datasets\\__init__.py'>
Version : 4.9.10
load existe : True
['GenerateMode', 'ImageFolder', 'ReadConfig', 'Split', 'TranslateFolder', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'annotations', 'as_dataframe', 'as_numpy', 'audio']


In [6]:
import tensorflow_datasets as tfds

(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)
print(ds_info)

tfds.core.DatasetInfo(
    name='imdb_reviews',
    full_name='imdb_reviews/plain_text/1.0.0',
    description="""
    Large Movie Review Dataset. This is a dataset for binary sentiment
    classification containing substantially more data than previous benchmark
    datasets. We provide a set of 25,000 highly polar movie reviews for training,
    and 25,000 for testing. There is additional unlabeled data for use as well.
    """,
    config_description="""
    Plain text
    """,
    homepage='http://ai.stanford.edu/~amaas/data/sentiment/',
    data_dir='C:\\Users\\meles\\tensorflow_datasets\\imdb_reviews\\plain_text\\1.0.0',
    file_format=tfrecord,
    download_size=80.23 MiB,
    dataset_size=129.83 MiB,
    features=FeaturesDict({
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
        'text': Text(shape=(), dtype=string),
    }),
    supervised_keys=('text', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'test': <

c:\Users\meles\Documents\TTA_DI_BootCamp_Gilles-Chris_MAKE\.venv\Lib\site-packages\tensorflow\python\data\ops\structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


In [7]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

Label: Negative
This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline ...

Label: Negative
I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film wa ...



In [8]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 1840cccf-8f04-45b2-99da-9eb94cfe1d72)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 7ac7dfec-3f21-4ad5-9adb-fc35a31a4726)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 

Tokenizer loaded: bert-base-uncased


In [9]:
MAX_LENGTH = 256
BATCH_SIZE = 16

if 'tokenizer' not in globals():
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
    print("Tokenizer loaded:", tokenizer.name_or_path)


def build_tensors(dataset, limit=None):
    texts = []
    labels = []

    iterator = dataset.take(limit) if limit is not None else dataset
    for example in iterator:
        text, label = example
        if hasattr(text, "numpy"):
            texts.append(text.numpy().decode("utf-8"))
        else:
            texts.append(str(text))
        labels.append(int(label.numpy()))

    encoded = tokenizer(
        texts,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf",
    )

    return {
        "input_ids": tf.cast(encoded["input_ids"], tf.int32),
        "attention_mask": tf.cast(encoded["attention_mask"], tf.int32),
        "token_type_ids": tf.cast(encoded["token_type_ids"], tf.int32),
    }, tf.constant(labels, dtype=tf.int32)


train_x, train_y = build_tensors(ds_train, limit=100)
test_x, test_y = build_tensors(ds_test, limit=20)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


In [10]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  109482240 
                                                                 
 dropout_37 (Dropout)        multiple                  0 (unused)
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
Total params: 109483778 (417.65 MB)
Trainable params: 109483778 (417.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [11]:
print(type(train_x))
print(train_x.keys())
print(train_x["input_ids"].shape)
print(train_x["attention_mask"].shape)
print(train_y.shape)

<class 'dict'>
dict_keys(['input_ids', 'attention_mask', 'token_type_ids'])
(100, 256)
(100, 256)
(100,)


In [12]:
if isinstance(model, tuple):
    model = model[0]

history = model.fit(
    x=train_x,
    y=train_y,
    validation_data=(test_x, test_y),
    batch_size=BATCH_SIZE,
    epochs=1,
)

1/7 [===>..........................] - ETA: 10:47 - loss: 0.6653 - accuracy: 0.5625

: 